# From Retrieval to Citation: The Effectiveness of Generative Engine Optimization Beyond the Generation Stage
#### RQ: How do content-based GEO edits affect the retrieval and generation stages of GSEs?

---
**Authors:** Leonard Rampf, Niklas Keckeisen

**Advisors:** Michail Batikas

**Submission Date:** 21.05.2026

## Objective

This notebook executes the retrieval-stage experiment against the Google Agent Search index populated in notebook `3_transform_to_jsonl`. For each of the 11,000 experimental units (500 queries × GEO editing dimensions), the original query is issued against the engine and the ranked retrieval results are recorded.

The experiment runs in up to three passes to recover from transient API errors, followed by an empty-query verification to diagnose persistent non-retrieval alerts. The raw results are then deduplicated and cleaned into a final dataset for statistical analysis.

**Output:** `20260413_retrieval_results_geo_v2.csv`

## Run 1: Initial Retrieval

Issues all 11,000 queries against the Agent Search engine. Anomalies are logged to an alert file for re-processing in subsequent runs.

In [ ]:
import json
import time
import os
import pandas as pd
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import clear_output

# 1. CONFIGURATION

PROJECT_ID = "project-6dc8c84c-e76e-4519-bb0" #insert your configuration. 
LOCATION = "global" #insert your configuration. 
ENGINE_ID = "retrieval-stage-geo-v2_1775725808007" #insert your configuration. 
JSONL_FILE_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_vertex_geoedits_v1.jsonl")
OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v1.csv")
ALERTS_OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v1.csv") 

client = discoveryengine.SearchServiceClient()
serving_config = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_config"

# 2. EXTRACT UNIQUE EXPERIMENTS

def get_unique_experiments(filepath):
    """Reads the JSONL and extracts unique query_id and original_query pairs."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            record = json.loads(line.strip())
            
            data.append({
                "query_id": record["structData"]["query_id"],
                "original_query": record["structData"]["original_query"]
            })
    
    df = pd.DataFrame(data)
    unique_experiments = df.drop_duplicates().reset_index(drop=True)
    return unique_experiments

# 3. THE RETRIEVAL FUNCTION

def run_query_and_get_all_results(search_query, query_id_filter):
    filter_str = f'query_id: ANY("{query_id_filter}")'
    threshold_enum = discoveryengine.SearchRequest.RelevanceThreshold
    
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=search_query,
        filter=filter_str,
        page_size=50,
        relevance_threshold=threshold_enum.LOWEST
    )

    results_for_this_query = []
    try:
        response = client.search(request)
        for index, result in enumerate(response.results):
            doc = result.document
            s_data = doc.struct_data
            d_data = doc.derived_struct_data
            doc_text = s_data.get("text") or d_data.get("text") or "N/A"

            results_for_this_query.append({
                "query_id": query_id_filter,
                "query": search_query,
                "doc_id": doc.id,
                "method": "default_method", 
                "doc_text": doc_text,
                "rank": index + 1,
                "is_target": 1 if "TARGET" in doc.id.upper() else 0
            })
        return results_for_this_query
    except Exception as e:
        return f"ERROR: {e}"

# 4. FULL EXECUTION LOOP

def execute_full_experiment():
    if os.path.exists(OUTPUT_PATH): os.remove(OUTPUT_PATH)
    if os.path.exists(ALERTS_OUTPUT_PATH): os.remove(ALERTS_OUTPUT_PATH)

    experiments_df = get_unique_experiments(JSONL_FILE_PATH)
    total_queries = len(experiments_df)
    
    # Trackers for the UI and Logging
    alerts_ui_display = []
    total_hits = 0
    total_processed = 0

    print(f"Starting background-safe run for {total_queries} queries...")

    try:
        for index, row in experiments_df.iterrows():
            q_id = row['query_id']
            o_query = row['original_query']
            
            query_results = run_query_and_get_all_results(o_query, q_id)
            
            # Temporary list to hold any alerts generated during this specific query
            current_query_alerts = []

            # 1. Handle API Errors
            if isinstance(query_results, str):
                current_query_alerts.append({
                    "query_id": q_id, "original_query": o_query,
                    "alert_type": "API_ERROR", "details": query_results
                })
                alerts_ui_display.append(f"API ERROR: {q_id}")
            else:
                num_found = len(query_results)
                target_found = any(r['is_target'] == 1 for r in query_results)
            
                # 2. Handle Diagnostics
                if num_found != 10:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "COUNT_ALERT", "details": f"Returned {num_found} docs"
                    })
                    alerts_ui_display.append(f"COUNT ALERT: {q_id} ({num_found} docs)")
                    
                if not target_found:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "TARGET_MISSING", "details": "Target not in top results"
                    })
                    alerts_ui_display.append(f"TARGET ALERT: {q_id}")

                # 3. Append Main Data to Disk
                if query_results:
                    batch_df = pd.DataFrame(query_results)
                    batch_df.to_csv(
                        OUTPUT_PATH, mode='a', index=False, 
                        header=not os.path.exists(OUTPUT_PATH), encoding='utf-8'
                    )

            # 4. Append Alerts to Log Disk
            if current_query_alerts:
                alerts_df = pd.DataFrame(current_query_alerts)
                alerts_df.to_csv(
                    ALERTS_OUTPUT_PATH, mode='a', index=False,
                    header=not os.path.exists(ALERTS_OUTPUT_PATH), encoding='utf-8'
                )

            total_processed += 1

            # 5. Notebook UI Update
            if index % 5 == 0 or index == total_queries - 1:
                clear_output(wait=True)
                print(f"Progress: [{index + 1}/{total_queries}] queries processed.")
                print(f"Total Alerts Triggered: {len(alerts_ui_display)}")
                
                if alerts_ui_display:
                    print("\nLatest 5 Alerts:")
                    for a in alerts_ui_display[-5:]:
                        print(a)
            
            time.sleep(0.1)

    except Exception as e:
        print(f"\nScript crashed unexpectedly: {e}")

    finally:
        print(f"\nFULL RUN COMPLETE")
        if os.path.exists(OUTPUT_PATH):
            file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
            print(f"Results File: {OUTPUT_PATH} ({file_size:.2f} MB)")
        else:
            print("No main results file created.")
            
        if os.path.exists(ALERTS_OUTPUT_PATH):
            alert_size = os.path.getsize(ALERTS_OUTPUT_PATH) / (1024) # KB
            print(f"Alerts File: {ALERTS_OUTPUT_PATH} ({alert_size:.2f} KB)")

# Run it
execute_full_experiment()

Progress: [96/11000] queries processed.
Total Alerts Triggered: 0


## Run 1: Alert Investigation

Run 1 produced 149 alerts: 1 API error (503 Bad Gateway), 128 incomplete responses (fewer than 10 documents returned), and 20 target-missing cases. As API errors and count mismatches may be transient, all 149 flagged queries are re-submitted in Run 2.

In [ ]:
# investigate alerts. If alerts such as API errors we run retrieval again as these might not happen a second time.
import pandas as pd
import os

# Load alerts
path = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v1.csv") 
alerts_df = pd.read_csv(path)

# 1. How many total alerts
unique_failing_queries = len(alerts_df['query_id'])

# 2. What are the top reasons for failure?
failure_summary = alerts_df.groupby(['alert_type', 'details']).size().reset_index(name='count')

print(f"Total Alerts: {unique_failing_queries}")
print("\n--- Failure Breakdown ---")
print(failure_summary)

Total Alerts: 149

--- Failure Breakdown ---
       alert_type                     details  count
0       API_ERROR  ERROR: 503 502:Bad Gateway      1
1     COUNT_ALERT             Returned 0 docs      4
2     COUNT_ALERT             Returned 8 docs      3
3     COUNT_ALERT             Returned 9 docs    121
4  TARGET_MISSING   Target not in top results     20


## Run 2: Error Recovery

Re-submits the 149 flagged queries from Run 1. The API error did not recur, and the alert count dropped from 149 to 138 — indicating some failures were transient. Further convergence is checked in Run 3.

In [ ]:
# second try to clear API errors etc. that happened during the first retrieval try

import json
import time
import os
import pandas as pd
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import clear_output

# 1. CONFIGURATION

PROJECT_ID = "project-6dc8c84c-e76e-4519-bb0" #insert your configuration. 
LOCATION = "global" #insert your configuration. 
ENGINE_ID = "retrieval-stage-geo-v2_1775725808007" #insert your configuration. 
CSV_FILE_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v1.csv")
OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v1.csv")
ALERTS_OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v2.csv") 

client = discoveryengine.SearchServiceClient()
serving_config = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_config"

# 2. EXTRACT UNIQUE EXPERIMENTS

def get_unique_experiments(filepath):
    print(f"Reading alerts from {filepath}...")
    df = pd.read_csv(filepath)
    unique_experiments = df[['query_id', 'original_query']].drop_duplicates().reset_index(drop=True)
    return unique_experiments

# 3. THE RETRIEVAL FUNCTION

def run_query_and_get_all_results(search_query, query_id_filter):
    filter_str = f'query_id: ANY("{query_id_filter}")'
    threshold_enum = discoveryengine.SearchRequest.RelevanceThreshold
    
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=search_query,
        filter=filter_str,
        page_size=50,
        relevance_threshold=threshold_enum.LOWEST
    )

    results_for_this_query = []
    try:
        response = client.search(request)
        for index, result in enumerate(response.results):
            doc = result.document
            s_data = doc.struct_data
            d_data = doc.derived_struct_data
            doc_text = s_data.get("text") or d_data.get("text") or "N/A"

            results_for_this_query.append({
                "query_id": query_id_filter,
                "query": search_query,
                "doc_id": doc.id,
                "method": "default_method", 
                "doc_text": doc_text,
                "rank": index + 1,
                "is_target": 1 if "TARGET" in doc.id.upper() else 0
            })
        return results_for_this_query
    except Exception as e:
        return f"ERROR: {e}"

# 4. FULL EXECUTION LOOP

def execute_full_experiment():

    experiments_df = get_unique_experiments(CSV_FILE_PATH)
    total_queries = len(experiments_df)
    
    # Trackers for the UI and Logging
    alerts_ui_display = []
    total_hits = 0
    total_processed = 0

    print(f"Starting background-safe run for {total_queries} queries...")

    try:
        for index, row in experiments_df.iterrows():
            q_id = row['query_id']
            o_query = row['original_query']
            
            query_results = run_query_and_get_all_results(o_query, q_id)
            
            # Temporary list to hold any alerts generated during this specific query
            current_query_alerts = []

            # 1. Handle API Errors
            if isinstance(query_results, str):
                current_query_alerts.append({
                    "query_id": q_id, "original_query": o_query,
                    "alert_type": "API_ERROR", "details": query_results
                })
                alerts_ui_display.append(f"API ERROR: {q_id}")
            else:
                num_found = len(query_results)
                target_found = any(r['is_target'] == 1 for r in query_results)
            
                # 2. Handle Diagnostics
                if num_found != 10:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "COUNT_ALERT", "details": f"Returned {num_found} docs"
                    })
                    alerts_ui_display.append(f"COUNT ALERT: {q_id} ({num_found} docs)")
                    
                if not target_found:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "TARGET_MISSING", "details": "Target not in top results"
                    })
                    alerts_ui_display.append(f"TARGET ALERT: {q_id}")

                # 3. Append Main Data to Disk
                if query_results:
                    batch_df = pd.DataFrame(query_results)
                    batch_df.to_csv(
                        OUTPUT_PATH, mode='a', index=False, 
                        header=not os.path.exists(OUTPUT_PATH), encoding='utf-8'
                    )

            # 4. Append Alerts to Log Disk
            if current_query_alerts:
                alerts_df = pd.DataFrame(current_query_alerts)
                alerts_df.to_csv(
                    ALERTS_OUTPUT_PATH, mode='a', index=False,
                    header=not os.path.exists(ALERTS_OUTPUT_PATH), encoding='utf-8'
                )

            total_processed += 1

            # 5. Notebook UI Update
            if index % 5 == 0 or index == total_queries - 1:
                clear_output(wait=True)
                print(f"Progress: [{index + 1}/{total_queries}] queries processed.")
                print(f"Total Alerts Triggered: {len(alerts_ui_display)}")
                
                if alerts_ui_display:
                    print("\nLatest 5 Alerts:")
                    for a in alerts_ui_display[-5:]:
                        print(a)
            
            time.sleep(0.1)

    except Exception as e:
        print(f"\nScript crashed unexpectedly: {e}")

    finally:
        print(f"\nFULL RUN COMPLETE")
        if os.path.exists(OUTPUT_PATH):
            file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
            print(f"Results File: {OUTPUT_PATH} ({file_size:.2f} MB)")
        else:
            print("No main results file created.")
            
        if os.path.exists(ALERTS_OUTPUT_PATH):
            alert_size = os.path.getsize(ALERTS_OUTPUT_PATH) / (1024) # KB
            print(f"Alerts File: {ALERTS_OUTPUT_PATH} ({alert_size:.2f} KB)")

# Run it
execute_full_experiment()

Progress: [1/129] queries processed.
Total Alerts Triggered: 1

Latest 5 Alerts:
COUNT ALERT: 79695_Statistics(doc) (9 docs)


## Run 2: Alert Investigation

Run 2 reduced alerts to 138 (123 count alerts, 15 target-missing), clearing the API error. Since the count has not yet stabilised, a third pass is conducted to confirm convergence.

In [ ]:
# investigate alerts again. 
import pandas as pd
import os

# Load alerts
path = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v2.csv") 
alerts_df = pd.read_csv(path)

# 1. How many total alerts
unique_failing_queries = len(alerts_df['query_id'])

# 2. What are the top reasons for failure?
failure_summary = alerts_df.groupby(['alert_type', 'details']).size().reset_index(name='count')

print(f"Total Alerts: {unique_failing_queries}")
print("\n--- Failure Breakdown ---")
print(failure_summary)

Total Alerts: 138

--- Failure Breakdown ---
       alert_type                    details  count
0     COUNT_ALERT            Returned 8 docs      2
1     COUNT_ALERT            Returned 9 docs    121
2  TARGET_MISSING  Target not in top results     15


## Run 3: Error Recovery

Re-submits the 138 remaining flagged queries from Run 2 to test whether further transient failures can be resolved.

In [ ]:
# third try to ensure that we covered all avoidable errors

import json
import time
import os
import pandas as pd
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import clear_output

# 1. CONFIGURATION

PROJECT_ID = "project-6dc8c84c-e76e-4519-bb0" #insert your configuration. 
LOCATION = "global" #insert your configuration. 
ENGINE_ID = "retrieval-stage-geo-v2_1775725808007" #insert your configuration. 
CSV_FILE_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v2.csv")
OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v1.csv")
ALERTS_OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v3.csv") 

client = discoveryengine.SearchServiceClient()
serving_config = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_config"

# 2. EXTRACT UNIQUE EXPERIMENTS

def get_unique_experiments(filepath):
    print(f"Reading alerts from {filepath}...")
    df = pd.read_csv(filepath)
    unique_experiments = df[['query_id', 'original_query']].drop_duplicates().reset_index(drop=True)
    return unique_experiments

# 3. THE RETRIEVAL FUNCTION

def run_query_and_get_all_results(search_query, query_id_filter):
    filter_str = f'query_id: ANY("{query_id_filter}")'
    threshold_enum = discoveryengine.SearchRequest.RelevanceThreshold
    
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=search_query,
        filter=filter_str,
        page_size=50,
        relevance_threshold=threshold_enum.LOWEST
    )

    results_for_this_query = []
    try:
        response = client.search(request)
        for index, result in enumerate(response.results):
            doc = result.document
            s_data = doc.struct_data
            d_data = doc.derived_struct_data
            doc_text = s_data.get("text") or d_data.get("text") or "N/A"

            results_for_this_query.append({
                "query_id": query_id_filter,
                "query": search_query,
                "doc_id": doc.id,
                "method": "default_method", 
                "doc_text": doc_text,
                "rank": index + 1,
                "is_target": 1 if "TARGET" in doc.id.upper() else 0
            })
        return results_for_this_query
    except Exception as e:
        return f"ERROR: {e}"

# 4. FULL EXECUTION LOOP

def execute_full_experiment():

    experiments_df = get_unique_experiments(CSV_FILE_PATH)
    total_queries = len(experiments_df)
    
    # Trackers for the UI and Logging
    alerts_ui_display = []
    total_hits = 0
    total_processed = 0

    print(f"Starting background-safe run for {total_queries} queries...")

    try:
        for index, row in experiments_df.iterrows():
            q_id = row['query_id']
            o_query = row['original_query']
            
            query_results = run_query_and_get_all_results(o_query, q_id)
            
            # Temporary list to hold any alerts generated during this specific query
            current_query_alerts = []

            # 1. Handle API Errors
            if isinstance(query_results, str):
                current_query_alerts.append({
                    "query_id": q_id, "original_query": o_query,
                    "alert_type": "API_ERROR", "details": query_results
                })
                alerts_ui_display.append(f"API ERROR: {q_id}")
            else:
                num_found = len(query_results)
                target_found = any(r['is_target'] == 1 for r in query_results)
            
                # 2. Handle Diagnostics
                if num_found != 10:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "COUNT_ALERT", "details": f"Returned {num_found} docs"
                    })
                    alerts_ui_display.append(f"COUNT ALERT: {q_id} ({num_found} docs)")
                    
                if not target_found:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "TARGET_MISSING", "details": "Target not in top results"
                    })
                    alerts_ui_display.append(f"TARGET ALERT: {q_id}")

                # 3. Append Main Data to Disk
                if query_results:
                    batch_df = pd.DataFrame(query_results)
                    batch_df.to_csv(
                        OUTPUT_PATH, mode='a', index=False, 
                        header=not os.path.exists(OUTPUT_PATH), encoding='utf-8'
                    )

            # 4. Append Alerts to Log Disk
            if current_query_alerts:
                alerts_df = pd.DataFrame(current_query_alerts)
                alerts_df.to_csv(
                    ALERTS_OUTPUT_PATH, mode='a', index=False,
                    header=not os.path.exists(ALERTS_OUTPUT_PATH), encoding='utf-8'
                )

            total_processed += 1

            # 5. Notebook UI Update
            if index % 5 == 0 or index == total_queries - 1:
                clear_output(wait=True)
                print(f"Progress: [{index + 1}/{total_queries}] queries processed.")
                print(f"Total Alerts Triggered: {len(alerts_ui_display)}")
                
                if alerts_ui_display:
                    print("\nLatest 5 Alerts:")
                    for a in alerts_ui_display[-5:]:
                        print(a)
            
            time.sleep(0.1)

    except Exception as e:
        print(f"\nScript crashed unexpectedly: {e}")

    finally:
        print(f"\nFULL RUN COMPLETE")
        if os.path.exists(OUTPUT_PATH):
            file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
            print(f"Results File: {OUTPUT_PATH} ({file_size:.2f} MB)")
        else:
            print("No main results file created.")
            
        if os.path.exists(ALERTS_OUTPUT_PATH):
            alert_size = os.path.getsize(ALERTS_OUTPUT_PATH) / (1024) # KB
            print(f"Alerts File: {ALERTS_OUTPUT_PATH} ({alert_size:.2f} KB)")

# Run it
execute_full_experiment()

Progress: [36/123] queries processed.
Total Alerts Triggered: 43

Latest 5 Alerts:
COUNT ALERT: 74158_SimpleLanguage(doc) (8 docs)
COUNT ALERT: 24586_Quotes(doc) (9 docs)
COUNT ALERT: 24586_FCS(doc) (9 docs)
COUNT ALERT: 39241_Citations(doc) (9 docs)
COUNT ALERT: 39241_LLMstxt(doc) (9 docs)


## Run 3: Alert Investigation

Alert count remained at 138 with an identical breakdown to Run 2, confirming convergence. The remaining anomalies are not transient. An empty-query verification is conducted next to determine whether they reflect indexing gaps or genuine relevance mismatches.

In [ ]:
# investigate alerts again. 
import pandas as pd
import os

# Load alerts
path = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v3.csv") 
alerts_df = pd.read_csv(path)

# 1. How many total alerts
unique_failing_queries = len(alerts_df['query_id'])

# 2. What are the top reasons for failure?
failure_summary = alerts_df.groupby(['alert_type', 'details']).size().reset_index(name='count')

print(f"Total Alerts: {unique_failing_queries}")
print("\n--- Failure Breakdown ---")
print(failure_summary)

Total Alerts: 138

--- Failure Breakdown ---
       alert_type                    details  count
0     COUNT_ALERT            Returned 8 docs      2
1     COUNT_ALERT            Returned 9 docs    121
2  TARGET_MISSING  Target not in top results     15


## Empty Query Verification

To distinguish indexing errors from relevance-based non-retrieval, the 138 flagged queries are re-issued with an empty search string, bypassing relevance scoring. If all 10 documents are returned, the documents exist in the index and the alerts reflect low query–document relevance rather than missing data.

In [ ]:
# now investigate the remaining 138 errors

import json
import time
import os
import pandas as pd
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import clear_output 

# 1. CONFIGURATION

PROJECT_ID = "project-6dc8c84c-e76e-4519-bb0" #insert your configuration. 
LOCATION = "global" #insert your configuration. 
ENGINE_ID = "retrieval-stage-geo-v2_1775725808007" #insert your configuration. 
INPUT_CSV_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v3.csv") 
OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_investigation_results_v1.csv")
ALERTS_OUTPUT_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_alert_log_geo_v4.csv") 

client = discoveryengine.SearchServiceClient()
serving_config = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_config"

# 2. EXTRACT UNIQUE EXPERIMENTS

def get_unique_experiments(filepath):
    print(f"Reading alerts from {filepath}...")
    df = pd.read_csv(filepath)
    unique_experiments = df[['query_id', 'original_query']].drop_duplicates().reset_index(drop=True)
    return unique_experiments

# 3. THE RETRIEVAL FUNCTION

def run_query_and_get_all_results(search_query, query_id_filter):
    filter_str = f'query_id: ANY("{query_id_filter}")'
    threshold_enum = discoveryengine.SearchRequest.RelevanceThreshold
    
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=search_query,
        filter=filter_str,
        page_size=50,
        relevance_threshold=threshold_enum.LOWEST 
    )

    results_for_this_query = []
    try:
        response = client.search(request)
        for index, result in enumerate(response.results):
            doc = result.document
            s_data = doc.struct_data
            d_data = doc.derived_struct_data
            doc_text = s_data.get("text") or d_data.get("text") or "N/A"

            results_for_this_query.append({
                "query_id": query_id_filter,
                "query": search_query,
                "doc_id": doc.id,
                "method": "empty_query_test", 
                "doc_text": doc_text,
                "rank": index + 1,
                "is_target": 1 if "TARGET" in doc.id.upper() else 0
            })
        return results_for_this_query
    except Exception as e:
        return f"ERROR: {e}"

# 4. FULL EXECUTION LOOP

def execute_investigation():
    if os.path.exists(OUTPUT_PATH): os.remove(OUTPUT_PATH)
    if os.path.exists(ALERTS_OUTPUT_PATH): os.remove(ALERTS_OUTPUT_PATH)

    experiments_df = get_unique_experiments(INPUT_CSV_PATH)
    total_queries = len(experiments_df)
    
    total_processed = 0

    print(f"Starting Empty Query Verification for {total_queries} queries...")

    try:
        for index, row in experiments_df.iterrows():
            q_id = row['query_id']
            # We keep the original query just for logging purposes
            o_query = row['original_query'] 
            
            # We force the search query to be empty to bypass relevance scoring
            forced_empty_query = ""
            
            query_results = run_query_and_get_all_results(forced_empty_query, q_id)
            current_query_alerts = []

            if isinstance(query_results, str):
                current_query_alerts.append({
                    "query_id": q_id, "original_query": o_query,
                    "alert_type": "API_ERROR", "details": query_results
                })
            else:
                num_found = len(query_results)
                target_found = any(r['is_target'] == 1 for r in query_results)
                
                
                # Diagnostics
                if num_found != 10:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "COUNT_ALERT", "details": f"Returned {num_found} docs"
                    })
                if not target_found:
                    current_query_alerts.append({
                        "query_id": q_id, "original_query": o_query,
                        "alert_type": "TARGET_MISSING", "details": "Target not in top results"
                    })

                if query_results:
                    batch_df = pd.DataFrame(query_results)
                    batch_df.to_csv(OUTPUT_PATH, mode='a', index=False, 
                                    header=not os.path.exists(OUTPUT_PATH), encoding='utf-8')

            if current_query_alerts:
                alerts_df = pd.DataFrame(current_query_alerts)
                alerts_df.to_csv(ALERTS_OUTPUT_PATH, mode='a', index=False,
                                 header=not os.path.exists(ALERTS_OUTPUT_PATH), encoding='utf-8')

            total_processed += 1

            if index % 5 == 0 or index == total_queries - 1:
                clear_output(wait=True)
                print(f"Empty Query Progress: [{index + 1}/{total_queries}]")
            
            time.sleep(0.1)

    except Exception as e:
        print(f"\nUnexpected Crash: {e}")
    finally:
        print(f"\nVerification Complete.")
        if os.path.exists(ALERTS_OUTPUT_PATH):
            final_alerts = pd.read_csv(ALERTS_OUTPUT_PATH)
            print(f"Total persisting data gaps: {len(final_alerts)}")

# Run it
execute_investigation()

Empty Query Progress: [123/123]

Verification Complete.


**Finding:** Re-running the 138 failing queries with an empty search string returned all 10 documents in every case. This confirms the documents exist in the index and are correctly linked to their `query_id`. The persistent alerts therefore reflect low relevance between the document text and the original query — a legitimate retrieval outcome, not a data or indexing error. No further error handling is applied.

## Dataset Cleanup

The three retrieval passes appended results sequentially to a single file, creating duplicate rows for re-submitted queries. This step retains only the most recent attempt per experimental unit, reducing 112,093 raw rows to 109,875 clean rows.

In [ ]:
# after the alert analysis we now clean up the dataset

import pandas as pd
import os

# 1. FILE CONFIGURATION

# The messy file containing Run 1 + Run 2 + Run 3
RAW_RESULTS_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v1.csv")

# The clean file
FINAL_CLEAN_PATH = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v2.csv")

def clean_master_dataset():
    print(f"Reading raw appended data from: {RAW_RESULTS_PATH}...")
    
    if not os.path.exists(RAW_RESULTS_PATH):
        print("Error: Raw results file not found.")
        return

    df = pd.read_csv(RAW_RESULTS_PATH)
    initial_row_count = len(df)
    print(f"Total raw rows: {initial_row_count}")

    # 2. THE CHUNKING LOGIC

    # Every time the query_id changes from one row to the next, 
    # it assigns a new "chunk_id". This groups contiguous attempts together.
    df['chunk_id'] = (df['query_id'] != df['query_id'].shift()).cumsum()
    
    # For every unique query_id, find the highest (i.e., the most recent) chunk_id
    latest_chunks = df.groupby('query_id')['chunk_id'].max()
    
    # Filter the dataframe to ONLY keep rows that belong to those final chunks
    df_clean = df[df['chunk_id'].isin(latest_chunks)].copy()
    
    # Clean up the temporary tracking column
    df_clean = df_clean.drop(columns=['chunk_id'])
    
    final_row_count = len(df_clean)
    rows_removed = initial_row_count - final_row_count
    
    # 3. VERIFICATION & SAVE

    print(f"Removed old/failed attempts: {rows_removed} rows discarded.")
    print(f"Final pristine rows: {final_row_count}")
    
    # Verification: Ensure no query_id has more than 10 documents
    max_docs_per_query = df_clean.groupby('query_id').size().max()
    print(f"Maximum documents per query in final set: {max_docs_per_query} (Should be ≤ 10)")
    
    if max_docs_per_query > 10:
        print("WARNING: Some queries have more than 10 documents. Check your search parameters.")
        
    df_clean.to_csv(FINAL_CLEAN_PATH, index=False, encoding='utf-8')
    print(f"\nSaved clean dataset for statistical analysis to: {FINAL_CLEAN_PATH}")

# Execute the cleanup
clean_master_dataset()

Reading raw appended data from: data/20260413_retrieval_results_geo_v1.csv...
Total raw rows: 112093
Removed old/failed attempts: 2218 rows discarded.
Final pristine rows: 109875
Maximum documents per query in final set: 10 (Should be ≤ 10)

Saved clean dataset for statistical analysis to: data/20260413_retrieval_results_geo_v2.csv
